In [2]:
%load_ext autoreload
%autoreload 2

import torch

from utils import create_edge_loaders, create_hetero_graph
from gnn import model_factory
from train import train, evaluate, gnn_evaluate

/home/jen55/.conda/envs/cpsc483/lib/python3.10/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: Could not load this library: /nfs/roberts/project/pi_cmu2/jen55/ycrc_conda/envs/cpsc483/lib/python3.10/site-packages/torch_scatter/_version_cuda.so
  import torch_geometric.typing
/home/jen55/.conda/envs/cpsc483/lib/python3.10/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-cluster'. Disabling its usage. Stacktrace: Could not load this library: /nfs/roberts/project/pi_cmu2/jen55/ycrc_conda/envs/cpsc483/lib/python3.10/site-packages/torch_cluster/_version_cuda.so
  import torch_geometric.typing
/home/jen55/.conda/envs/cpsc483/lib/python3.10/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-sparse'. Disabling its usage. Stacktrace: Could not load this library: /nfs/roberts/project/pi_cmu2/jen55/ycrc_conda

In [3]:
# Follow and activity data paths
FOLLOW_EDGELIST = 'data/higgs-social_network.edgelist'
ACTIVITY_EDGELIST = 'data/higgs-activity_time.txt'
target_interaction = "mention"

torch.manual_seed(42)

In [3]:
model_path = train(
    model_name="sage", 
    target_interaction=target_interaction, 
    activity_data_path=ACTIVITY_EDGELIST, 
    follow_data_path=FOLLOW_EDGELIST,
    max_epochs=10,
    checkpoint_dir=f'checkpoints_{target_interaction}'
)

/home/jen55/.conda/envs/cpsc483/lib/python3.10/site-packages/lightning_fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/jen55/.conda/envs/cpsc483/lib/python3.10/site- ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
/home/jen55/.conda/envs/cpsc483/lib/python3.10/site-packages/pytorch_lightning/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch_lightning` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
💡 Tip: For seamless cloud 

Starting training...
Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/jen55/.conda/envs/cpsc483/lib/python3.10/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Epoch 9: 100%|██████████| 937/937 [00:29<00:00, 31.25it/s, v_num=82, train_loss_step=0.619, train_acc_step=0.708, val_loss=0.790, val_auroc=0.715, val_acc=0.620, val_ap=0.627, val_recall=0.722, train_loss_epoch=0.631, train_acc_epoch=0.712]

`Trainer.fit` stopped: `max_epochs=10` reached.


Epoch 9: 100%|██████████| 937/937 [00:29<00:00, 31.24it/s, v_num=82, train_loss_step=0.619, train_acc_step=0.708, val_loss=0.790, val_auroc=0.715, val_acc=0.620, val_ap=0.627, val_recall=0.722, train_loss_epoch=0.631, train_acc_epoch=0.712]
Training complete. Best model saved at: /nfs/roberts/scratch/pi_cmu2/jen55/GraphFinalProject/checkpoints_mention/best-gnn-sage-mentionepoch=02-val_ap=0.6700.ckpt


In [4]:
# Evaluation
best_model_path = model_path
# best_model_path = "/nfs/roberts/scratch/pi_cmu2/jen55/GraphFinalProject/checkpoints/best-gnn-sage-followepoch=01-val_ap=0.8651.ckpt"
print(f"Evaluating best model from: {best_model_path}")

evaluate(
    checkpoint_path=best_model_path,
    model_name="sage",
    target_interaction=target_interaction,
    activity_data_path=ACTIVITY_EDGELIST,
    follow_data_path=FOLLOW_EDGELIST,
    visualize_limit=0
)

Evaluating best model from: /nfs/roberts/scratch/pi_cmu2/jen55/GraphFinalProject/checkpoints_mention/best-gnn-sage-mentionepoch=02-val_ap=0.6700.ckpt
Using device: cuda
Loading data...
Configuring CAPTUM Integrated Gradients...


Evaluating & Explaining (Captum):   0%|          | 0/201 [00:00<?, ?it/s]/nfs/roberts/scratch/pi_cmu2/jen55/GraphFinalProject/train/evaluate.py:164: UserWarning: The 'target' should not be provided for the explanation type 'model'
  explanation = explainer(
Evaluating & Explaining (Captum): 100%|██████████| 201/201 [01:40<00:00,  2.00it/s]


--- Standard Metrics ---
Loss:   0.8241
AUROC:  0.8005
AP:     0.6826
Recall: 0.8503

--- Explanation Metrics (over 201 batches) ---
Saved visualization files to current directory.
Average Fidelity+: 0.6667 (Higher is better)
Average Fidelity-: 0.4527 (Lower is better)


In [5]:
# Using GNNExplainer
# best_model_path = "/nfs/roberts/scratch/pi_cmu2/jen55/GraphFinalProject/checkpoints/best-gnn-sage-followepoch=01-val_ap=0.8651.ckpt"
gnn_evaluate(
    checkpoint_path=best_model_path,
    model_name="sage",
    target_interaction=target_interaction,
    activity_data_path=ACTIVITY_EDGELIST,
    follow_data_path=FOLLOW_EDGELIST,
    explain_limit=5
)

Using device: cuda
Loading data...

--- Evaluating Test Set ---


Evaluating: 100%|██████████| 201/201 [00:02<00:00, 76.73it/s]



--- Standard Metrics ---
Loss:  0.8234
AUROC: 0.8008
AP:    0.6830

--- Setting up Explainer (Option B) ---


Explaining:   0%|          | 0/5 [00:00<?, ?it/s]


--- Top Explanations for Target Edge 0 ---
  Top 5 important 'FL' edges:
    User 245 -> User 23 (Weight: 0.8501)
    User 19 -> User 23 (Weight: 0.1391)
    User 19 -> User 15 (Weight: 0.1343)
    User 21 -> User 19 (Weight: 0.1342)
    User 3 -> User 6 (Weight: 0.1333)
  Top 5 important 'RT' edges:
    User 288 -> User 23 (Weight: 0.8988)
    User 274 -> User 5 (Weight: 0.0831)
    User 259 -> User 2 (Weight: 0.0830)
    User 280 -> User 5 (Weight: 0.0830)
    User 281 -> User 5 (Weight: 0.0830)
  Top 5 important 'MT' edges:
    User 23 -> User 1 (Weight: 0.8015)
    User 1 -> User 23 (Weight: 0.1001)
    User 316 -> User 23 (Weight: 0.0989)
    User 317 -> User 23 (Weight: 0.0899)
    User 315 -> User 23 (Weight: 0.0880)
  Top 5 important 'RE' edges:
    User 325 -> User 2 (Weight: 0.9332)
    User 320 -> User 2 (Weight: 0.9331)
    User 267 -> User 2 (Weight: 0.9331)
    User 23 -> User 1 (Weight: 0.8950)
    User 1 -> User 23 (Weight: 0.1060)


...


--- Top Explanations for Target Edge 2 ---
  Top 5 important 'FL' edges:
    User 541 -> User 63 (Weight: 0.1631)
    User 901 -> User 63 (Weight: 0.1590)
    User 51 -> User 1 (Weight: 0.1495)
    User 40 -> User 1 (Weight: 0.1477)
    User 32 -> User 1 (Weight: 0.1474)
  Top 5 important 'RT' edges:
    User 43 -> User 1 (Weight: 0.2342)
    User 668 -> User 46 (Weight: 0.1057)
    User 914 -> User 31 (Weight: 0.1055)
    User 1013 -> User 52 (Weight: 0.1055)
    User 929 -> User 34 (Weight: 0.1051)
  Top 5 important 'MT' edges:
    User 62 -> User 0 (Weight: 0.1286)
    User 63 -> User 0 (Weight: 0.1241)
    User 1045 -> User 10 (Weight: 0.0985)
    User 1048 -> User 24 (Weight: 0.0978)
    User 908 -> User 24 (Weight: 0.0977)
  Top 5 important 'RE' edges:
    User 63 -> User 0 (Weight: 0.0929)
    User 1051 -> User 25 (Weight: 0.0816)
    User 1056 -> User 33 (Weight: 0.0816)
    User 1067 -> User 45 (Weight: 0.0816)
    User 1072 -> User 45 (Weight: 0.0816)


Explaining:  80%|████████  | 4/5 [00:10<00:02,  2.54s/it]


--- Top Explanations for Target Edge 3 ---
  Top 5 important 'FL' edges:
    User 238 -> User 8 (Weight: 0.1500)
    User 551 -> User 30 (Weight: 0.1468)
    User 654 -> User 38 (Weight: 0.1464)
    User 321 -> User 14 (Weight: 0.1463)
    User 775 -> User 116 (Weight: 0.1462)
  Top 5 important 'RT' edges:
    User 1076 -> User 59 (Weight: 0.0920)
    User 71 -> User 1 (Weight: 0.0918)
    User 90 -> User 1 (Weight: 0.0918)
    User 82 -> User 1 (Weight: 0.0916)
    User 88 -> User 1 (Weight: 0.0916)
  Top 5 important 'MT' edges:
    User 100 -> User 1 (Weight: 0.0878)
    User 97 -> User 1 (Weight: 0.0877)
    User 119 -> User 1 (Weight: 0.0877)
    User 1104 -> User 102 (Weight: 0.0874)
    User 1094 -> User 21 (Weight: 0.0874)
  Top 5 important 'RE' edges:
    User 1100 -> User 86 (Weight: 0.0788)
    User 122 -> User 1 (Weight: 0.0786)
    User 97 -> User 1 (Weight: 0.0786)
    User 119 -> User 1 (Weight: 0.0786)
    User 1090 -> User 11 (Weight: 0.0786)


Explaining: 100%|██████████| 5/5 [00:13<00:00,  2.61s/it]


--- Top Explanations for Target Edge 4 ---
  Top 5 important 'FL' edges:
    User 756 -> User 48 (Weight: 0.1508)
    User 39 -> User 1 (Weight: 0.1481)
    User 1060 -> User 105 (Weight: 0.1475)
    User 589 -> User 34 (Weight: 0.1474)
    User 871 -> User 58 (Weight: 0.1473)
  Top 5 important 'RT' edges:
    User 1137 -> User 21 (Weight: 0.1084)
    User 1144 -> User 31 (Weight: 0.1076)
    User 1110 -> User 11 (Weight: 0.1066)
    User 1123 -> User 21 (Weight: 0.1066)
    User 1172 -> User 40 (Weight: 0.1065)
  Top 5 important 'MT' edges:
    User 1214 -> User 11 (Weight: 0.1026)
    User 1228 -> User 11 (Weight: 0.1022)
    User 1238 -> User 21 (Weight: 0.1015)
    User 93 -> User 0 (Weight: 0.1012)
    User 1211 -> User 9 (Weight: 0.1012)
  Top 5 important 'RE' edges:
    User 115 -> User 0 (Weight: 0.0844)
    User 1273 -> User 53 (Weight: 0.0839)
    User 1276 -> User 76 (Weight: 0.0839)
    User 1291 -> User 32 (Weight: 0.0838)
    User 1292 -> User 32 (Weight: 0.0838)

--- Ex

In [4]:
# Comparison with vanilla GNN
vanilla_model_path = train(
    model_name="simple",
    target_interaction=target_interaction,
    activity_data_path=ACTIVITY_EDGELIST,
    follow_data_path=FOLLOW_EDGELIST,
    max_epochs=10,
    checkpoint_dir=f'checkpoints_{target_interaction}'
)

/home/jen55/.conda/envs/cpsc483/lib/python3.10/site-packages/lightning_fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/jen55/.conda/envs/cpsc483/lib/python3.10/site- ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
/home/jen55/.conda/envs/cpsc483/lib/python3.10/site-packages/pytorch_lightning/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch_lightning` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
💡 Tip: For seamless cloud 

Starting training...
Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/jen55/.conda/envs/cpsc483/lib/python3.10/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Epoch 9: 100%|██████████| 937/937 [00:57<00:00, 16.20it/s, v_num=98, train_loss_step=0.635, train_acc_step=0.860, val_loss=0.673, val_auroc=0.658, val_acc=0.765, val_ap=0.530, val_recall=0.336, train_loss_epoch=0.611, train_acc_epoch=0.850]

`Trainer.fit` stopped: `max_epochs=10` reached.


Epoch 9: 100%|██████████| 937/937 [00:57<00:00, 16.19it/s, v_num=98, train_loss_step=0.635, train_acc_step=0.860, val_loss=0.673, val_auroc=0.658, val_acc=0.765, val_ap=0.530, val_recall=0.336, train_loss_epoch=0.611, train_acc_epoch=0.850]
Training complete. Best model saved at: /nfs/roberts/scratch/pi_cmu2/jen55/GraphFinalProject/checkpoints_mention/best-gnn-simple-mentionepoch=00-val_ap=0.5531.ckpt


In [9]:
# vanilla_model_path = "/nfs/roberts/scratch/pi_cmu2/jen55/GraphFinalProject/checkpoints_mention/best-gnn-simple-mentionepoch=00-val_ap=0.5531.ckpt"
print(f"Evaluating vanilla GNN model from: {vanilla_model_path}")
evaluate(
    checkpoint_path=vanilla_model_path,
    model_name="simple",
    target_interaction=target_interaction,
    activity_data_path=ACTIVITY_EDGELIST,
    follow_data_path=FOLLOW_EDGELIST,
    visualize_limit=5
)
gnn_evaluate(
    checkpoint_path=vanilla_model_path,
    model_name="simple",
    target_interaction=target_interaction,
    activity_data_path=ACTIVITY_EDGELIST,
    follow_data_path=FOLLOW_EDGELIST,
    explain_limit=5
)

Evaluating vanilla GNN model from: /nfs/roberts/scratch/pi_cmu2/jen55/GraphFinalProject/checkpoints_mention/best-gnn-simple-mentionepoch=00-val_ap=0.5531.ckpt
Using device: cuda
Loading data...
Configuring CAPTUM Integrated Gradients...


Evaluating & Explaining (Captum):   0%|          | 0/201 [00:00<?, ?it/s]/nfs/roberts/scratch/pi_cmu2/jen55/GraphFinalProject/train/evaluate.py:164: UserWarning: The 'target' should not be provided for the explanation type 'model'
  explanation = explainer(
Evaluating & Explaining (Captum):   1%|          | 2/201 [00:04<08:09,  2.46s/it]/nfs/roberts/scratch/pi_cmu2/jen55/GraphFinalProject/train/evaluate.py:164: UserWarning: The 'target' should not be provided for the explanation type 'model'
  explanation = explainer(
Evaluating & Explaining (Captum):   2%|▏         | 4/201 [00:11<12:10,  3.71s/it]/nfs/roberts/scratch/pi_cmu2/jen55/GraphFinalProject/train/evaluate.py:164: UserWarning: The 'target' should not be provided for the explanation type 'model'
  explanation = explainer(
Evaluating & Explaining (Captum):   2%|▏         | 5/201 [00:18<15:37,  4.79s/it]/nfs/roberts/scratch/pi_cmu2/jen55/GraphFinalProject/train/evaluate.py:164: UserWarning: The 'target' should not be provided for 


--- Standard Metrics ---
Loss:   0.7008
AUROC:  0.6951
AP:     0.5652
Recall: 0.4300

--- Explanation Metrics (over 201 batches) ---
Saved visualization files to current directory.
Average Fidelity+: 0.3582 (Higher is better)
Average Fidelity-: 0.2786 (Lower is better)
Using device: cuda
Loading data...

--- Evaluating Test Set ---


Evaluating: 100%|██████████| 201/201 [00:03<00:00, 59.96it/s] 



--- Standard Metrics ---
Loss:  0.7036
AUROC: 0.6947
AP:    0.5634

--- Setting up Explainer (Option B) ---


...


--- Top Explanations for Target Edge 1 ---
  Top 5 important 'FL' edges:
    User 158 -> User 18 (Weight: 0.1460)
    User 563 -> User 83 (Weight: 0.1431)
    User 509 -> User 69 (Weight: 0.1425)
    User 52 -> User 1 (Weight: 0.1424)
    User 561 -> User 83 (Weight: 0.1422)
  Top 5 important 'RT' edges:
    User 83 -> User 1 (Weight: 0.1035)
    User 62 -> User 1 (Weight: 0.1018)
    User 69 -> User 1 (Weight: 0.1006)
    User 652 -> User 22 (Weight: 0.0995)
    User 68 -> User 1 (Weight: 0.0991)
  Top 5 important 'MT' edges:
    User 101 -> User 1 (Weight: 0.9361)
    User 688 -> User 92 (Weight: 0.0839)
    User 99 -> User 1 (Weight: 0.0817)
    User 97 -> User 1 (Weight: 0.0812)
    User 702 -> User 92 (Weight: 0.0812)
  Top 5 important 'RE' edges:
    User 696 -> User 61 (Weight: 0.9294)
    User 701 -> User 84 (Weight: 0.9293)
    User 92 -> User 1 (Weight: 0.9292)
    User 92 -> User 1 (Weight: 0.9292)
    User 93 -> User 1 (Weight: 0.0732)


Explaining:  60%|██████    | 3/5 [00:04<00:03,  1.51s/it]


--- Top Explanations for Target Edge 2 ---
  Top 5 important 'FL' edges:
    User 784 -> User 63 (Weight: 0.2853)
    User 403 -> User 63 (Weight: 0.2808)
    User 19 -> User 0 (Weight: 0.2031)
    User 23 -> User 0 (Weight: 0.1984)
    User 25 -> User 0 (Weight: 0.1966)
  Top 5 important 'RT' edges:
    User 33 -> User 1 (Weight: 0.5699)
    User 947 -> User 36 (Weight: 0.1055)
    User 959 -> User 37 (Weight: 0.1050)
    User 912 -> User 32 (Weight: 0.1049)
    User 897 -> User 32 (Weight: 0.1048)
  Top 5 important 'MT' edges:
    User 63 -> User 0 (Weight: 0.3493)
    User 62 -> User 0 (Weight: 0.1753)
    User 1010 -> User 23 (Weight: 0.0984)
    User 1017 -> User 31 (Weight: 0.0971)
    User 1015 -> User 31 (Weight: 0.0971)
  Top 5 important 'RE' edges:
    User 63 -> User 0 (Weight: 0.0819)
    User 1042 -> User 34 (Weight: 0.0761)
    User 1073 -> User 44 (Weight: 0.0760)
    User 1096 -> User 32 (Weight: 0.0760)
    User 1009 -> User 18 (Weight: 0.0760)

--- Top Explanations f

Explaining: 100%|██████████| 5/5 [00:11<00:00,  2.20s/it]


--- Top Explanations for Target Edge 4 ---
  Top 5 important 'FL' edges:
    User 752 -> User 50 (Weight: 0.1499)
    User 39 -> User 1 (Weight: 0.1486)
    User 1062 -> User 107 (Weight: 0.1481)
    User 936 -> User 81 (Weight: 0.1480)
    User 558 -> User 82 (Weight: 0.1478)
  Top 5 important 'RT' edges:
    User 78 -> User 0 (Weight: 0.1002)
    User 1156 -> User 50 (Weight: 0.1002)
    User 63 -> User 0 (Weight: 0.1000)
    User 59 -> User 0 (Weight: 0.0997)
    User 1160 -> User 50 (Weight: 0.0996)
  Top 5 important 'MT' edges:
    User 1196 -> User 28 (Weight: 0.0967)
    User 1185 -> User 5 (Weight: 0.0963)
    User 1206 -> User 32 (Weight: 0.0962)
    User 975 -> User 87 (Weight: 0.0960)
    User 1223 -> User 51 (Weight: 0.0958)
  Top 5 important 'RE' edges:
    User 972 -> User 87 (Weight: 0.0818)
    User 1221 -> User 50 (Weight: 0.0818)
    User 1224 -> User 53 (Weight: 0.0818)
    User 1188 -> User 24 (Weight: 0.0818)
    User 557 -> User 44 (Weight: 0.0817)

--- Explanati

In [4]:
# Ablation study: stripped GNN
stripped_model_path = train(
    model_name="stripped",
    target_interaction=target_interaction,
    activity_data_path=ACTIVITY_EDGELIST,
    follow_data_path=FOLLOW_EDGELIST,
    max_epochs=10,
    checkpoint_dir=f'checkpoints_{target_interaction}'
)

/home/jen55/.conda/envs/cpsc483/lib/python3.10/site-packages/lightning_fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/jen55/.conda/envs/cpsc483/lib/python3.10/site- ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
/home/jen55/.conda/envs/cpsc483/lib/python3.10/site-packages/pytorch_lightning/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch_lightning` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
💡 Tip: For seamless cloud 

Starting training...
Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/jen55/.conda/envs/cpsc483/lib/python3.10/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Epoch 9: 100%|██████████| 937/937 [00:23<00:00, 40.03it/s, v_num=102, train_loss_step=0.605, train_acc_step=0.795, val_loss=0.664, val_auroc=0.592, val_acc=0.726, val_ap=0.455, val_recall=0.188, train_loss_epoch=0.621, train_acc_epoch=0.782]

`Trainer.fit` stopped: `max_epochs=10` reached.


Epoch 9: 100%|██████████| 937/937 [00:23<00:00, 40.02it/s, v_num=102, train_loss_step=0.605, train_acc_step=0.795, val_loss=0.664, val_auroc=0.592, val_acc=0.726, val_ap=0.455, val_recall=0.188, train_loss_epoch=0.621, train_acc_epoch=0.782]
Training complete. Best model saved at: /nfs/roberts/scratch/pi_cmu2/jen55/GraphFinalProject/checkpoints_mention/best-gnn-stripped-mentionepoch=00-val_ap=0.5245.ckpt


In [10]:
# stripped_model_path = "/nfs/roberts/scratch/pi_cmu2/jen55/GraphFinalProject/checkpoints_mention/best-gnn-stripped-mentionepoch=00-val_ap=0.5245.ckpt"
print(f"Evaluating stripped GNN model from: {stripped_model_path}")
evaluate(
    checkpoint_path=stripped_model_path,
    model_name="stripped",
    target_interaction=target_interaction,
    activity_data_path=ACTIVITY_EDGELIST,
    follow_data_path=FOLLOW_EDGELIST,
    visualize_limit=5
)
gnn_evaluate(
    checkpoint_path=stripped_model_path,
    model_name="stripped",
    target_interaction=target_interaction,
    activity_data_path=ACTIVITY_EDGELIST,
    follow_data_path=FOLLOW_EDGELIST,
    explain_limit=5
)

Evaluating stripped GNN model from: /nfs/roberts/scratch/pi_cmu2/jen55/GraphFinalProject/checkpoints_mention/best-gnn-stripped-mentionepoch=00-val_ap=0.5245.ckpt
Using device: cuda
Loading data...
Configuring CAPTUM Integrated Gradients...


Evaluating & Explaining (Captum):   0%|          | 0/201 [00:00<?, ?it/s]/nfs/roberts/scratch/pi_cmu2/jen55/GraphFinalProject/train/evaluate.py:164: UserWarning: The 'target' should not be provided for the explanation type 'model'
  explanation = explainer(
Evaluating & Explaining (Captum):   2%|▏         | 4/201 [00:01<00:58,  3.37it/s]/nfs/roberts/scratch/pi_cmu2/jen55/GraphFinalProject/train/evaluate.py:164: UserWarning: The 'target' should not be provided for the explanation type 'model'
  explanation = explainer(
Evaluating & Explaining (Captum):   2%|▏         | 5/201 [00:01<01:05,  2.97it/s]/nfs/roberts/scratch/pi_cmu2/jen55/GraphFinalProject/train/evaluate.py:164: UserWarning: The 'target' should not be provided for the explanation type 'model'
  explanation = explainer(
Evaluating & Explaining (Captum): 100%|██████████| 201/201 [00:37<00:00,  5.41it/s]



--- Standard Metrics ---
Loss:   1.0664
AUROC:  0.7300
AP:     0.5341
Recall: 0.6576

--- Explanation Metrics (over 201 batches) ---
Saved visualization files to current directory.
Average Fidelity+: 0.5274 (Higher is better)
Average Fidelity-: 0.5373 (Lower is better)
Using device: cuda
Loading data...

--- Evaluating Test Set ---


Evaluating: 100%|██████████| 201/201 [00:03<00:00, 63.58it/s] 



--- Standard Metrics ---
Loss:  1.0602
AUROC: 0.7311
AP:    0.5360

--- Setting up Explainer (Option B) ---


...


--- Top Explanations for Target Edge 2 ---
  Top 5 important 'FL' edges:
    User 5 -> User 0 (Weight: 0.0878)
    User 17 -> User 0 (Weight: 0.0878)
    User 35 -> User 1 (Weight: 0.0876)
    User 43 -> User 1 (Weight: 0.0875)
    User 46 -> User 1 (Weight: 0.0875)
  Top 5 important 'RT' edges:
    User 62 -> User 1 (Weight: 0.0723)
    User 937 -> User 8 (Weight: 0.0000)
    User 935 -> User 8 (Weight: 0.0000)
    User 938 -> User 12 (Weight: 0.0000)
    User 936 -> User 8 (Weight: 0.0000)
  Top 5 important 'MT' edges:
    User 64 -> User 0 (Weight: 0.9286)
    User 63 -> User 0 (Weight: 0.0725)
    User 1094 -> User 12 (Weight: 0.0000)
    User 1093 -> User 6 (Weight: 0.0000)
    User 1092 -> User 6 (Weight: 0.0000)
  Top 5 important 'RE' edges:
    User 64 -> User 0 (Weight: 0.9283)
    User 253 -> User 15 (Weight: 0.0000)
    User 1095 -> User 12 (Weight: 0.0000)
    User 1197 -> User 32 (Weight: 0.0000)
    User 1095 -> User 12 (Weight: 0.0000)


Explaining:  80%|████████  | 4/5 [00:03<00:00,  1.17it/s]


--- Top Explanations for Target Edge 3 ---
  Top 5 important 'FL' edges:
    User 16 -> User 0 (Weight: 0.0882)
    User 42 -> User 1 (Weight: 0.0878)
    User 59 -> User 1 (Weight: 0.0877)
    User 7 -> User 0 (Weight: 0.0876)
    User 41 -> User 1 (Weight: 0.0875)
  Top 5 important 'RT' edges:
    User 87 -> User 1 (Weight: 0.0800)
    User 89 -> User 1 (Weight: 0.0799)
    User 76 -> User 1 (Weight: 0.0799)
    User 79 -> User 1 (Weight: 0.0799)
    User 68 -> User 1 (Weight: 0.0799)
  Top 5 important 'MT' edges:
    User 108 -> User 1 (Weight: 0.0800)
    User 113 -> User 1 (Weight: 0.0799)
    User 112 -> User 1 (Weight: 0.0799)
    User 97 -> User 1 (Weight: 0.0799)
    User 107 -> User 1 (Weight: 0.0798)
  Top 5 important 'RE' edges:
    User 98 -> User 1 (Weight: 0.0749)
    User 118 -> User 1 (Weight: 0.0749)
    User 121 -> User 1 (Weight: 0.0748)
    User 96 -> User 1 (Weight: 0.0748)
    User 101 -> User 1 (Weight: 0.0748)


Explaining: 100%|██████████| 5/5 [00:04<00:00,  1.16it/s]


--- Top Explanations for Target Edge 4 ---
  Top 5 important 'FL' edges:
    User 46 -> User 1 (Weight: 0.0866)
    User 16 -> User 0 (Weight: 0.0866)
    User 38 -> User 1 (Weight: 0.0865)
    User 4 -> User 0 (Weight: 0.0865)
    User 8 -> User 0 (Weight: 0.0864)
  Top 5 important 'RT' edges:
    User 69 -> User 0 (Weight: 0.0800)
    User 58 -> User 0 (Weight: 0.0800)
    User 67 -> User 0 (Weight: 0.0799)
    User 57 -> User 0 (Weight: 0.0799)
    User 78 -> User 0 (Weight: 0.0798)
  Top 5 important 'MT' edges:
    User 104 -> User 0 (Weight: 0.0799)
    User 92 -> User 0 (Weight: 0.0799)
    User 97 -> User 0 (Weight: 0.0799)
    User 100 -> User 0 (Weight: 0.0799)
    User 105 -> User 0 (Weight: 0.0799)
  Top 5 important 'RE' edges:
    User 115 -> User 0 (Weight: 0.9287)
    User 117 -> User 0 (Weight: 0.0727)
    User 116 -> User 0 (Weight: 0.0727)
    User 1257 -> User 32 (Weight: 0.0000)
    User 1219 -> User 23 (Weight: 0.0000)

--- Explanation Metrics (over 5 edges) ---
Av